# 14 — Compare: Qasper (kvpress vs vLLM)

Side-by-side comparison of KV cache compression on the
[Qasper](https://huggingface.co/datasets/tau/scrolls) benchmark (SCROLLS)
from notebooks 09 (kvpress) and 10 (vLLM).

Both frameworks test **KeyDiffPress**-based compression with two
decoding strategies:
- **full_replacement** — full KV cache replacement after prefill
- **filtering** — token filtering during decoding

Qasper is a single document QA task scored with SQuAD F1 —
no subtask breakdown, so we use line charts instead of heatmaps.

Visualizations:
1. F1 and Exact Match vs compression ratio (line charts)
2. Summary table

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

ALGORITHMS = ['full_replacement', 'filtering']
SHORT_NAMES = {
    'full_replacement': 'Full Replacement',
    'filtering': 'Filtering',
    'no_press': 'No compression',
}

## 1. Load Results

In [ ]:
def load_qasper_metrics(path, framework):
    with open(path) as f:
        raw = json.load(f)
    rows = []
    for key, scores in raw.items():
        parts = key.split('__')
        press = parts[0]
        ratio = float(parts[1])
        rows.append({
            'framework': framework,
            'press': press,
            'compression_ratio': ratio,
            'f1': scores['f1'],
            'exact_match': scores['exact_match'],
        })
    return pd.DataFrame(rows)

kvpress_df = load_qasper_metrics('results/kvpress_qasper/metrics.json', 'kvpress')
vllm_df = load_qasper_metrics('results/vllm_qasper/metrics.json', 'vllm')
df = pd.concat([kvpress_df, vllm_df], ignore_index=True)

print(f'Loaded {len(kvpress_df)} kvpress + {len(vllm_df)} vllm = {len(df)} total')
print(f'Frameworks: {sorted(df["framework"].unique())}')
print(f'Algorithms: {sorted(df["press"].unique())}')
print(df.to_string(index=False))

## 2. F1 and Exact Match vs Compression Ratio

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = {'kvpress': 'tab:blue', 'vllm': 'tab:orange'}
styles = {'full_replacement': '-', 'filtering': '--'}
markers = {'full_replacement': 'o', 'filtering': 's'}

for metric, ax, title in [('f1', ax1, 'F1'), ('exact_match', ax2, 'Exact Match')]:
    for fw in ['kvpress', 'vllm']:
        for algo in ALGORITHMS:
            sub = df[(df['framework'] == fw) & (df['press'] == algo)].sort_values('compression_ratio')
            if sub.empty:
                continue
            ax.plot(
                sub['compression_ratio'], sub[metric],
                marker=markers[algo], linewidth=2,
                color=colors[fw], linestyle=styles[algo],
                label=f'{fw} — {SHORT_NAMES[algo]}',
            )

    for fw in ['kvpress', 'vllm']:
        baseline = df[(df['framework'] == fw) & (df['press'] == 'no_press')]
        if not baseline.empty:
            score = baseline[metric].values[0]
            ax.axhline(
                score, color=colors[fw], linestyle=':', linewidth=1.5, alpha=0.5,
                label=f'{fw} — No compression ({score:.1f})',
            )

    ax.set_xlabel('Compression Ratio')
    ax.set_ylabel(f'{title} (%)')
    ax.set_title(f'Qasper: {title} vs Compression Ratio')
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig('results/compare_qasper_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Summary

In [ ]:
summary = df.sort_values(['framework', 'press', 'compression_ratio'])
print(summary.to_string(index=False))